# 🚀 Step 2 (vLLM): Qwen3.8-27B AWQ + ngrok 公開APIサーバー

`twolven/Qwen3.8-27B-Uncensored-HauhauCS-Aggressive-AWQ-MTP` を **vLLM** で T4 x2 に載せ、ngrok で公開します。

---

### 前提条件（これだけ）

1. **Accelerator**: `GPU T4 x2`
2. **Internet**: `On`
3. **Add Data**: 以下を追加
   - `yamahei21python/qwen38-27b-uncensored-hauhaucs-awq-mtp`（AWQ 4bit モデル 約19 GB）

> ✅ **必要な Secrets**: `NGROK_TOKEN`（必須）
> `NGROK_DOMAIN` はノートに埋め込み済み → **毎回同じ URL**。
>
> 🔓 認証はありません（URL を知れば誰でも使えます）。
> 他人に使われたくないときは環境変数 `API_PASS` を設定してください。

---

### Ollama 版との違い

| 項目 | Ollama 版 | このノート（vLLM） |
|---|---|---|
| 量子化 | GGUF Q4_K_M | AWQ W4A16 |
| 推論 | llama.cpp | vLLM（PagedAttention） |
| MTP | なし | `--speculative-config` で有効化（失敗時は自動で無効） |
| 速度 | 13.7 t/s 実測 | T4 x2 では 10〜25 t/s 見込み |

> ℹ T4 は sm75 のため **Marlin カーネルと FP8 KV cache は使えません**。
> AWQ は Triton 実装にフォールバックします。


---
### ⚙️ 設定（Dataset 検出 → VRAM 計測 → 起動条件決定）


In [ ]:
# ============================================================
# Step 2 (vLLM): 共通設定
# ============================================================
import glob as _glob
import json
import os
import re
import subprocess
import time


def _find_input(slug):
    """Dataset のマウント場所を探す。

    Kaggle は /kaggle/input/<slug> と /kaggle/input/datasets/<owner>/<slug> の
    両方が実在しうる。深さ2まで探索し、最初に「空でない」ものを返す。
    """
    cands = []
    for pat in ("/kaggle/input/" + slug,
                "/kaggle/input/*/" + slug,
                "/kaggle/input/*/*/" + slug):
        cands += [p for p in _glob.glob(pat) if os.path.isdir(p)]
    for p in sorted(cands):
        try:
            if os.listdir(p):
                return p
        except OSError:
            continue
    return cands[0] if cands else "/kaggle/input/" + slug


MODEL_SRC = _find_input("qwen38-27b-uncensored-hauhaucs-awq-mtp")
SERVED_NAME = "qwen3.8-27b-awq"
VLLM_PORT = 8000
PROXY_PORT = 11435
TUNNEL_KIND = "ngrok"

# Basic 認証プロキシ / UI のソース（ビルダーが埋め込む）
PROXY_SRC_B64 = "..."
CHAT_SRC_B64 = "..."
PROXY_PATH = "/usr/local/bin/auth_proxy.py"
CHAT_PATH = "/usr/local/bin/llama-chat.html"

# pip に渡す vLLM 指定（切り戻したいときは "vllm==0.10.2" 等に変える）
VLLM_PIP_SPEC = os.environ.get("VLLM_PIP_SPEC", "vllm")


def _detect_gpu():
    """全 GPU の (名前, VRAM MiB) を返す。"""
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total",
             "--format=csv,noheader"],
            capture_output=True, text=True, timeout=15,
        ).stdout.strip()
        names, mems = [], []
        for line in out.splitlines():
            if not line.strip():
                continue
            parts = line.split(",")
            names.append(parts[0].strip())
            mems.append(int(re.sub(r"[^0-9]", "", parts[1])))
        return names, mems
    except Exception as e:
        print(f"GPU 検出失敗: {e}")
        return [], []


_gpu_names, _vram_each = _detect_gpu()
VRAM_TOTAL = sum(_vram_each)
print(f"GPU      : {_gpu_names}")
print(f"VRAM     : {_vram_each} MiB each  →  合計 {VRAM_TOTAL} MiB")

if VRAM_TOTAL == 0:
    raise RuntimeError(
        "GPU が見つかりません。Accelerator を 'GPU T4 x2' に設定してください。")
if len(_vram_each) < 2:
    print("⚠ GPU が 1 枚です。Tensor Parallel は 1 に落とします"
          "（27B AWQ は 1 枚に収まらない可能性が高い）")
TP_SIZE = 2 if len(_vram_each) >= 2 else 1

# ---- 起動条件 ----
# 27B AWQ は約 19 GB。T4 x2 = 30 GB のうち 0.90 = 27 GB を使う想定で、
# 残り ~8 GB が KV cache になる。実際に収まるかは config 次第なので、
# 起動失敗時に max-model-len を落として再試行する（serve セル）。
MAXLEN_CANDIDATES = [32768, 16384, 8192]
MAX_MODEL_LEN = MAXLEN_CANDIDATES[0]
GPU_MEM_UTIL = 0.90
MAX_NUM_SEQS = 2
KV_CACHE_DTYPE = "auto"        # T4 は FP8 非対応。fp8 を指定してはいけない

# MTP（multi-token prediction）は任意。失敗したら自動で無効にして再起動する。
MTP_ENABLED = os.environ.get("VLLM_MTP", "1") != "0"
MTP_METHOD = os.environ.get("VLLM_SPEC_METHOD", "mtp")
MTP_TOKENS = int(os.environ.get("VLLM_SPEC_TOKENS", "2"))

# ---- 公開ポート ----
API_USER = os.environ.get("API_USER", "") or "ollama"
API_PASS = os.environ.get("API_PASS", "")

# ---- 永続化ヘルパー ----
# `kaggle kernels output` が zip 化するのは /kaggle/working のみ。
OUT_DIRS = ("/kaggle/working", "/output")


def save_output(name, text):
    """検証成果物を永続領域へ。1 つでも書ければ True（成功条件に使う）。"""
    saved = []
    for d in OUT_DIRS:
        try:
            os.makedirs(d, exist_ok=True)
            with open(os.path.join(d, name), "w") as f:
                f.write(text)
            saved.append(d)
        except Exception as e:
            print(f"  ⚠ {d} 書き出し失敗 ({e})")
    if saved:
        print(f"  📄 {name} 保存済: {', '.join(saved)}")
        return True
    print(f"  ⚠ 成果物を1つも保存できませんでした: {name}")
    return False


def fail(msg, tail_path=None):
    """失敗原因を永続領域に残してから raise（ログ API は当てにできない）。"""
    tail = ""
    try:
        if tail_path and os.path.isfile(tail_path):
            tail = open(tail_path, errors="replace").read()[-3000:]
    except Exception:
        pass
    save_output("FAILED.txt", f"{msg}\n\n{tail}")
    raise RuntimeError(msg)


print()
print("=" * 56)
print(f"  モデル      : {MODEL_SRC}")
print(f"  配信名      : {SERVED_NAME}")
print(f"  TP          : {TP_SIZE}")
print(f"  MTP         : {'ON' if MTP_ENABLED else 'OFF'}")
print(f"  公開ポート  : {PROXY_PORT}（vLLM 本体は {VLLM_PORT}）")
print(f"  認証        : {'ON' if API_PASS else 'OFF（URL を知れば誰でも利用可）'}")
print("=" * 56)

_files = sorted(os.listdir(MODEL_SRC)) if os.path.isdir(MODEL_SRC) else []
print(f"  Dataset 直下: {len(_files)} 件")
for _f in _files[:12]:
    print(f"    - {_f}")
if "config.json" not in _files:
    fail("config.json が Dataset 直下にありません（Add Data を確認）")
if "model.safetensors" not in _files:
    fail("model.safetensors が Dataset 直下にありません（Add Data を確認）")


---
### 📡 vLLM インストール + プロキシ配置

vLLM の wheel 取得に 5〜10 分かかります（2回目以降は Kaggle のキャッシュで短縮）。


In [ ]:
# vLLM をインストールし、プロキシと UI を書き出す
import base64
import os
import shutil
import subprocess
import sys

BIN_DIR = "/usr/local/bin"
MIN_SIZE = 1024 * 1024

# ---- cloudflared（Dataset キャッシュ優先）----
if TUNNEL_KIND != "ngrok":
    CF_PATH = _find_input("my-cloudflared-bin")
    dst = os.path.join(BIN_DIR, "cloudflared")
    src = None
    for cand in (os.path.join(CF_PATH, "cloudflared"),
                 os.path.join(CF_PATH, "bin", "cloudflared")):
        if os.path.isfile(cand) and os.path.getsize(cand) >= MIN_SIZE:
            src = cand
            break
    if src:
        shutil.copy2(src, dst)
        subprocess.run(["chmod", "+x", dst])
        print(f"✅ cloudflared 復元: {src}")
    elif not (os.path.isfile(dst) and os.path.getsize(dst) >= MIN_SIZE):
        print("⬇️  cloudflared をダウンロード中...")
        subprocess.run(
            "curl -fsSL --connect-timeout 15 -o /usr/local/bin/cloudflared "
            "https://github.com/cloudflare/cloudflared/releases/latest/download/"
            "cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared",
            shell=True, check=False, timeout=300,
        )
    if not os.path.isfile(dst):
        fail("cloudflared がありません（Dataset 未マウント / 取得失敗）")
    r = subprocess.run([dst, "--version"], capture_output=True, text=True)
    print((r.stdout or r.stderr).strip())
    if r.returncode != 0:
        fail("cloudflared の実行に失敗しました（実行ビット/Dataset を確認）")

# ---- vLLM ----
_r = subprocess.run([sys.executable, "-c", "import vllm"],
                    capture_output=True, text=True)
if _r.returncode == 0:
    print("✅ vLLM は既にインストール済み")
else:
    print(f"⬇️  vLLM をインストール中: {VLLM_PIP_SPEC}（5〜10分）")
    _p = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-input",
         VLLM_PIP_SPEC],
        capture_output=True, text=True, timeout=3600,
    )
    print((_p.stdout or "")[-2000:])
    print((_p.stderr or "")[-2000:])
    if _p.returncode != 0:
        fail("vLLM のインストールに失敗しました（上のログ参照）")

# torchaudio は transformers が無条件に import する。vLLM の pip install で
# torch だけが差し替わると CUDA 版数が不一致になり、vllm CLI が即死する
# （実測: vllm 0.32.0 / torch 2.13.0+cu130 に対し torchaudio 2.8.3+cu128）。
# 起動ラダーに入る前にここで直す。直せなければ早期に止める。
_PREFLIGHT = ("import vllm.entrypoints.cli.main;"
              "from transformers import AutoConfig")


def _cli_ok():
    r = subprocess.run([sys.executable, "-c", _PREFLIGHT],
                       capture_output=True, text=True, timeout=600)
    return r.returncode == 0, (r.stderr or "")[-1200:]


_ok, _err = _cli_ok()
if not _ok:
    print("⚠ vLLM CLI の import に失敗 → torchaudio を揃える")
    print(_err)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-input",
                    "-U", "torchaudio"], capture_output=True, text=True,
                   timeout=3600)
    _ok, _err = _cli_ok()
if not _ok:
    print("⚠ torchaudio 更新でも直らない → torchaudio を外して再試行")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y",
                    "torchaudio"], capture_output=True, text=True, timeout=600)
    _ok, _err = _cli_ok()
if not _ok:
    fail("vLLM CLI の import に失敗（torch/torchaudio 不整合の疑い）: " + _err)

try:
    import torch
    import vllm
except Exception as _e:
    # import 失敗は「torch の入れ替わり」で起きやすい。原因を残して止める。
    fail(f"vllm/torch の import に失敗: {type(_e).__name__}: {_e}")
print(f"✅ vLLM {vllm.__version__} / torch {torch.__version__} "
      f"/ CUDA {torch.version.cuda}")
print(f"   GPU 認識: {torch.cuda.device_count()} 枚")

# ---- プロキシ + UI ----
with open(PROXY_PATH, "wb") as f:
    f.write(base64.b64decode(PROXY_SRC_B64))
with open(CHAT_PATH, "wb") as f:
    f.write(base64.b64decode(CHAT_SRC_B64))
print(f"✅ auth_proxy.py: {os.path.getsize(PROXY_PATH):,} B")
print(f"✅ llama-chat.html: {os.path.getsize(CHAT_PATH):,} B")


---
### 🌐 リバースプロキシ起動（UI 配信 + 同一オリジン化）

vLLM 本体は 8000、公開は 11435。`/v1/*` をそのまま中継します。


In [ ]:
# リバースプロキシを起動（UI を / で配信し、API を同一オリジンにする）
import os
import subprocess
import time
import urllib.error
import urllib.request

subprocess.run(["pkill", "-f", "auth_proxy.py"], capture_output=True)
time.sleep(1)

proxy_env = {
    **os.environ,
    # 認証は既定で無効（API_PASS を設定したときだけ有効）
    "API_USER": API_USER,
    "API_PASS": API_PASS,
}
with open("/tmp/auth_proxy.log", "w") as log:
    proxy_proc = subprocess.Popen(
        ["python3", PROXY_PATH,
         "--port", str(PROXY_PORT),
         "--upstream", f"http://127.0.0.1:{VLLM_PORT}",
         "--chat-html", CHAT_PATH,
         # クライアントが temperature=0 / repetition_penalty 未指定で送っても
         # 崩壊しないよう、サーバ側で下限を矯正する（RP>=1.05 / temp>0）。
         "--enforce-sampling"],
        env=proxy_env, stdout=log, stderr=log,
    )
print(f"🔄 proxy PID: {proxy_proc.pid}")

for i in range(30):
    try:
        with urllib.request.urlopen(
            f"http://127.0.0.1:{PROXY_PORT}/healthz", timeout=3
        ) as r:
            if r.status == 200:
                print(f"✅ プロキシ起動完了（{i+1}s）")
                break
    except Exception:
        pass
    time.sleep(1)
else:
    print(open("/tmp/auth_proxy.log").read()[-2000:])
    fail("認証プロキシの起動に失敗しました", "/tmp/auth_proxy.log")

# vLLM はまだ起動していないので 502 で正常（401 なら設定ミス）
req = urllib.request.Request(f"http://127.0.0.1:{PROXY_PORT}/v1/models")
try:
    with urllib.request.urlopen(req, timeout=10) as r:
        print(f"  ✅ 認証なし → HTTP {r.status}")
except urllib.error.HTTPError as e:
    if e.code in (401, 403):
        fail(f"認証なしで HTTP {e.code} — 認証が有効になっています")
    print(f"  ✅ 認証なし → HTTP {e.code}（上流未起動。401 でない）")
except Exception as e:
    fail(f"プロキシ検査に失敗: {type(e).__name__}: {e}")


---
### 🔥 vLLM サーバー起動（初回は 5〜10 分）

起動条件は **上から順に試行**し、失敗したら次へ落ちます。

| # | MTP | max-model-len | 備考 |
|---|---|---|---|
| 1 | ON | 32768 | 本命 |
| 2 | ON | 16384 | KV 不足時 |
| 3 | OFF | 16384 | MTP 非対応アーキのとき |
| 4 | OFF | 8192 | 最小構成 |
| 5 | OFF | 8192 | V0 エンジン（T4 用フォールバック） |

`--speculative-config` などのフラグは `vllm serve --help` を見て
**存在するものだけ**付けます（バージョン差で即死しないため）。


In [ ]:
# vLLM 起動 → /v1/models が返るまで待機（失敗時は条件を落として再試行）
import json
import os
import re
import shutil
import subprocess
import time
import urllib.request

STAGE = "/kaggle/tmp/vllm-model"
MTP_FILE = "model-mtp.safetensors"


def _stage_model(with_mtp):
    """モデルを symlink で 1 か所にまとめる（19 GB をコピーしない）。

    MTP の draft head (model-mtp.safetensors) は、speculative を切った状態で
    読ませると unexpected key になるため、その場合だけ除外する。
    """
    if os.path.isdir(STAGE):
        shutil.rmtree(STAGE)          # symlink のみなので実体は消えない
    os.makedirs(STAGE, exist_ok=True)
    n = 0
    for fn in sorted(os.listdir(MODEL_SRC)):
        if fn.startswith("."):
            continue
        if fn == MTP_FILE and not with_mtp:
            continue
        src = os.path.join(MODEL_SRC, fn)
        if not os.path.isfile(src):
            continue
        os.symlink(src, os.path.join(STAGE, fn))
        n += 1
    return n


# ---- 利用可能なフラグを確認（バージョン差の吸収）----
_help = subprocess.run(["vllm", "serve", "--help"],
                       capture_output=True, text=True, timeout=180)
HELP = (_help.stdout or "") + (_help.stderr or "")
# ヘルプ本文を成果物に残す（フラグ判定が外れたときの唯一の手掛かり）
save_output("vllm_help.txt", HELP[:20000])
print(f"  vllm serve --help: exit={_help.returncode} {len(HELP)} 文字")
print(HELP[:1200])
HELP_OK = len(HELP) > 500


def _has(flag):
    """ヘルプ本文の「行頭オプション」として存在するか。

    単純な部分文字列一致だと説明文中の言及で誤判定するため行頭で見る。
    ヘルプが取れない環境（バージョン差で別コマンドに変わった等）では
    「ある」とみなす。存在しないフラグはその試行が即座に落ちるだけで、
    次の段（任意フラグ off）で拾えるため、楽観側に倒すほうが損が小さい。
    """
    if not HELP_OK:
        return True
    return re.search(r"^\s*" + re.escape(flag) + r"\b", HELP,
                     re.MULTILINE) is not None


# 必須フラグ（ここが通らなければ何を試しても無駄）
CORE_ARGS = [
    "vllm", "serve", STAGE,
    "--served-model-name", SERVED_NAME,
    "--host", "127.0.0.1",
    "--port", str(VLLM_PORT),
    "--tensor-parallel-size", str(TP_SIZE),
    "--dtype", "half",
    "--quantization", "awq",
    "--max-num-seqs", str(MAX_NUM_SEQS),
    "--gpu-memory-utilization", str(GPU_MEM_UTIL),
    "--kv-cache-dtype", KV_CACHE_DTYPE,
]

# 任意フラグ。バージョンによって引数値が違うことがあるので、
# 後半の試行では落として「フラグが原因の失敗」を切り分ける。
OPT_ARGS = []
if _has("--trust-remote-code"):
    OPT_ARGS.append("--trust-remote-code")
if _has("--disable-log-requests"):
    OPT_ARGS.append("--disable-log-requests")
if _has("--reasoning-parser"):
    # Qwen3 系の思考タグを reasoning_content に分離する
    OPT_ARGS += ["--reasoning-parser", "qwen3"]
if _has("--enable-auto-tool-choice") and _has("--tool-call-parser"):
    OPT_ARGS += ["--enable-auto-tool-choice", "--tool-call-parser", "hermes"]

# ---- 試行計画（上から順に）----
# 各要素: (ラベル, speculative config, max-model-len, 追加env, 任意フラグを使うか)
SPEC = None
if MTP_ENABLED and _has("--speculative-config"):
    SPEC = {"method": MTP_METHOD, "num_speculative_tokens": MTP_TOKENS}
else:
    print("ℹ --speculative-config 非対応 or MTP=0 → MTP なしで起動")

# V0 エンジン（T4 フォールバック）。V1 が sm75 非対応の vLLM で効く。
V0_ENV = {"VLLM_USE_V1": "0", "VLLM_ATTENTION_BACKEND": "XFORMERS"}

PLAN = []
if SPEC:
    PLAN.append(("mtp", SPEC, MAXLEN_CANDIDATES[0], {}, True, []))
    PLAN.append(("mtp", SPEC, MAXLEN_CANDIDATES[1], {}, True, []))
# ここからは任意フラグを落として最小構成で試す
PLAN.append(("plain", None, MAXLEN_CANDIDATES[0], {}, False, []))
PLAN.append(("plain", None, MAXLEN_CANDIDATES[1], {}, False, []))
PLAN.append(("plain", None, 8192, {}, False, []))
# V0 エンジン（T4 フォールバック）
PLAN.append(("v0", None, 8192, V0_ENV, False, []))
# CUDA グラフ取得が T4 で落ちる場合の最終手段（遅いが確実に動く）
PLAN.append(("eager", None, 8192, {}, False, ["--enforce-eager"]))


def _kill(proc):
    """vLLM は子プロセス（EngineCore）を抱えるので、木ごと落とす。"""
    try:
        proc.terminate()
        proc.wait(timeout=30)
    except Exception:
        try:
            proc.kill()
        except Exception:
            pass
    # fork された EngineCore が残ると VRAM を掴んだままになる
    subprocess.run(["pkill", "-9", "-f", "VLLM::EngineCore"],
                   capture_output=True)
    subprocess.run(["pkill", "-9", "-f", "vllm serve"], capture_output=True)
    time.sleep(3)


def _models(timeout=8):
    """起動済みなら /v1/models の JSON を返す（未起動なら None）。"""
    try:
        with urllib.request.urlopen(
            f"http://127.0.0.1:{VLLM_PORT}/v1/models", timeout=timeout
        ) as r:
            return json.loads(r.read())
    except Exception:
        return None


proc = None
argv_used = None
started = False
# GPU クォータは有限。ラダーを全部試すと 1 時間超になり、途中で打ち切られると
# 成果物が残らない。総予算を切って、必ずログを残して終わるようにする。
STARTUP_BUDGET = 1500          # 秒（25 分）
ATTEMPT_TIMEOUT = 600          # 1 試行あたり（10 分）
_attempts = []
_t_start = time.time()
save_output("vllm_attempts.txt", "起動試行ログ（追記保存）\n")

for idx, (label, spec, maxlen, env_extra, use_opt, extra_args) in enumerate(PLAN, 1):
    _elapsed = time.time() - _t_start
    if _elapsed > STARTUP_BUDGET:
        print(f"⏱ 起動予算 {STARTUP_BUDGET}s 超過（{int(_elapsed)}s）"
              f" → 残り {len(PLAN) - idx + 1} 試行を打ち切り")
        break
    n_staged = _stage_model(with_mtp=spec is not None)
    cmd = CORE_ARGS + (OPT_ARGS if use_opt else [])
    cmd += ["--max-model-len", str(maxlen)]
    if spec:
        cmd += ["--speculative-config", json.dumps(spec)]
    cmd += extra_args
    log_path = f"/tmp/vllm_{idx}.log"
    env_run = {**os.environ, **env_extra}
    print()
    print(f"--- 試行 {idx}/{len(PLAN)}: spec={label} "
          f"max_len={maxlen} env={env_extra or 'default'} "
          f"opt_flags={'on' if use_opt else 'off'} "
          f"staged={n_staged} files ---")
    with open(log_path, "w") as log:
        proc = subprocess.Popen(cmd, stdout=log, stderr=log, env=env_run)
    print(f"🔄 vLLM PID: {proc.pid}  log: {log_path}")

    t0 = time.time()
    while time.time() - t0 < ATTEMPT_TIMEOUT:
        d = _models(timeout=3)
        if d and d.get("data"):
            started = True
            break
        if proc.poll() is not None:
            break
        if int(time.time() - t0) % 60 < 5:
            print(f"  ⏳ 起動待ち {int(time.time()-t0)}s "
                  f"（{len(d.get('data', [])) if d else 0} models）")
        time.sleep(5)

    if started:
        argv_used = cmd
        print(f"✅ vLLM 起動完了（{int(time.time()-t0)}s, 試行 {idx}）")
        break

    tail = ""
    try:
        tail = open(log_path, errors="replace").read()[-4000:]
    except Exception:
        pass
    print("❌ 起動失敗。ログ末尾:")
    print(tail)
    # 途中で GPU クォータが尽きても原因が残るよう、失敗ごとに追記保存する。
    _attempts.append(f"--- 試行 {idx} {label} max_len={maxlen} "
                     f"opt={'on' if use_opt else 'off'} ---\n{tail}")
    save_output("vllm_attempts.txt", "\n\n".join(_attempts) + "\n")
    _kill(proc)

if not started:
    save_output("vllm_startup.log", tail)
    save_output("vllm_attempts.txt", "\n\n".join(_attempts) + "\n")
    fail("vLLM の起動に失敗しました（vllm_startup.log / vllm_*.log 参照）",
         "/tmp/vllm_%d.log" % len(PLAN))

# ---- 起動結果を保存 ----
info = _models() or {}
ids = [m.get("id") for m in info.get("data", [])]
max_len = (info.get("data") or [{}])[0].get("max_model_len")
with open("/tmp/vllm_cmd.json", "w") as f:
    json.dump({"argv": argv_used, "spec": bool(SPEC), "models": ids,
               "max_model_len": max_len}, f, indent=2)
print()
print(f"  models        : {ids}")
print(f"  max_model_len : {max_len}")
print(subprocess.run(
    ["nvidia-smi", "--query-gpu=memory.used,memory.total,utilization.gpu",
     "--format=csv,noheader,nounits"],
    capture_output=True, text=True).stdout.strip())
save_output("vllm_cmd.json", json.dumps(
    {"argv": argv_used, "models": ids, "max_model_len": max_len},
    indent=2) + "\n")


---
### 🔥 ウォームアップ（初回生成は CUDA graph 構築で遅い）


In [ ]:
# ダミー生成で疎通確認 + 速度計測
import json
import subprocess
import time
import urllib.request

payload = {
    "model": SERVED_NAME,
    "messages": [{"role": "user", "content": "Say OK."}],
    "max_tokens": 64,
    "temperature": 0.7,
    "repetition_penalty": 1.05,
    "stream": False,
}
req = urllib.request.Request(
    f"http://127.0.0.1:{VLLM_PORT}/v1/chat/completions",
    data=json.dumps(payload).encode(),
    headers={"Content-Type": "application/json"},
    method="POST",
)
print("⏳ ウォームアップ中...")
t0 = time.time()
try:
    with urllib.request.urlopen(req, timeout=900) as r:
        resp = json.loads(r.read())
    dt = time.time() - t0
except Exception as e:
    # 起動ログは試行番号で名前が変わる。読めなくても失敗自体は伝える。
    for _i in range(1, 7):
        _p = "/tmp/vllm_%d.log" % _i
        try:
            print(f"--- {_p} ---")
            print(open(_p, errors="replace").read()[-2500:])
        except OSError:
            continue
    fail(f"ウォームアップ失敗: {type(e).__name__}: {e}")

msg = (resp.get("choices") or [{}])[0].get("message") or {}
content = msg.get("content") or ""
reasoning = msg.get("reasoning_content") or ""
usage = resp.get("usage") or {}
ct = usage.get("completion_tokens") or 0
print(f"✅ 応答あり（{dt:.1f}s）")
print(f"   reply : {content[:120]!r}")
print(f"   think : {reasoning[:80]!r}")
print(f"   tokens: completion={ct} total={usage.get('total_tokens')}")
if ct:
    print(f"   速度  : {ct/dt:.1f} tokens/s（ウォームアップ込み）")
if not (content or reasoning or ct):
    fail("生成結果が空です（プロンプト/チャットテンプレートを確認）")

# GPU 常駐（CPU フォールバックでも HTTP 200 は返るので必ず見る）
mem = subprocess.run(
    ["nvidia-smi", "--query-gpu=memory.used,memory.total",
     "--format=csv,noheader,nounits"],
    capture_output=True, text=True).stdout.strip()
print(f"   VRAM  : {mem}")
used = [int(x.split(",")[0]) for x in mem.splitlines() if "," in x]
if used and sum(used) < 4000:
    fail(f"VRAM 使用量が少なすぎます（{used} MiB）。CPU 実行の可能性。")


---
### 🔗 ngrok トンネル → 公開URL

Secrets の `NGROK_TOKEN` を使います。`NGROK_DOMAIN` を設定すると
**毎回同じ URL** になります（未設定ならランダム）。


In [ ]:
# ngrok 起動 → 公開URL取得（authtoken は Secrets の NGROK_TOKEN）
import json, os, subprocess, time, urllib.request

NGROK_BIN = "/usr/local/bin/ngrok"
NGROK_API = "http://127.0.0.1:4040/api/tunnels"

# patch Dataset には入れていないので取得する（約 30 MB / 数秒）
if not os.path.isfile(NGROK_BIN):
    subprocess.run(
        ["bash", "-c",
         "curl -sSL https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz"
         " | tar xz -C /usr/local/bin"],
        check=True)

# ---- authtoken ----
_tok = os.environ.get("NGROK_TOKEN", "")
if not _tok:
    try:
        from kaggle_secrets import UserSecretsClient
        _tok = UserSecretsClient().get_secret("NGROK_TOKEN") or ""
    except Exception:
        _tok = ""
if not _tok:
    fail("NGROK_TOKEN がありません（Kaggle Secrets に登録してください）")
subprocess.run([NGROK_BIN, "config", "add-authtoken", _tok],
               check=True, capture_output=True)

# ---- 固定ドメイン ----
# 優先順: 環境変数 NGROK_DOMAIN → Secrets NGROK_DOMAIN → 埋め込み既定値。
NGROK_DOMAIN_DEFAULT = "rebalance-cornflake-sagging.ngrok-free.dev"
NGROK_DOMAIN = os.environ.get("NGROK_DOMAIN", "").strip()
if not NGROK_DOMAIN:
    try:
        from kaggle_secrets import UserSecretsClient
        NGROK_DOMAIN = (UserSecretsClient().get_secret("NGROK_DOMAIN") or "").strip()
    except Exception:
        NGROK_DOMAIN = ""
if not NGROK_DOMAIN:
    NGROK_DOMAIN = NGROK_DOMAIN_DEFAULT
print("📌 固定ドメイン: " + (NGROK_DOMAIN or "（未設定 → ランダムURL）"))

TUNNEL_NAME = "ngrok"
TUNNEL_LOG = "/tmp/ngrok.log"
# --pooling-enabled が無いと、同じ固定ドメインが前回セッションから残っている
# ときに ERR_NGROK_334 で即死する（実測）。
RESTART_CMD = [NGROK_BIN, "http", str(PROXY_PORT),
               "--log=stdout", "--pooling-enabled"]
if NGROK_DOMAIN:
    RESTART_CMD.append("--url=https://" + NGROK_DOMAIN)


def probe_url():
    """ngrok のローカル管理APIから公開URLを取る（ログ解析より確実）。"""
    try:
        with urllib.request.urlopen(NGROK_API, timeout=3) as r:
            d = json.loads(r.read().decode())
        for t in d.get("tunnels", []):
            u = t.get("public_url", "")
            if u.startswith("https://"):
                return u
    except Exception:
        pass
    return ""


subprocess.run(["pkill", "-x", "ngrok"], capture_output=True)
time.sleep(1)

with open(TUNNEL_LOG, "w") as _log:
    ng_proc = subprocess.Popen(RESTART_CMD, stdout=_log, stderr=_log)
print("🔄 ngrok PID: " + str(ng_proc.pid))

public_url = ""
for i in range(60):
    public_url = probe_url()
    if public_url:
        break
    if i % 10 == 9:
        print("⏳ URL 待ち (" + str(i + 1) + "/60)...")
    time.sleep(1)

if not public_url:
    print(open(TUNNEL_LOG).read()[-3000:])
    fail("ngrok の公開URL取得に失敗しました", TUNNEL_LOG)

print("🌍 URL: " + public_url)
print("⏳ トンネル確立待ち（最大 60s）...")
import urllib.error
ready = False
for i in range(20):
    try:
        with urllib.request.urlopen(public_url + "/healthz", timeout=10) as r:
            if r.status == 200:
                print("✅ トンネル確立（" + str((i + 1) * 3) + "s）")
                ready = True
                break
    except Exception:
        pass
    time.sleep(3)

if not ready:
    print("⚠ 確立を確認できませんでした。verify セルで再確認します。")

with open("/tmp/public_url.txt", "w") as f:
    f.write(public_url)
save_output("public_url.txt", public_url + "\n")
print()
print("=" * 60)
print("  🌍 PUBLIC URL : " + public_url)
print("=" * 60)


---
### 📌 固定URL配布（quick tunnel の URL を Gist に反映）

quick tunnel の URL は毎回変わるので、**取得先だけ固定**します。
`GITHUB_TOKEN` 未設定ならスキップします。


In [ ]:
# 現在の公開URLを Gist に書き出す（取得先の固定）
# keepalive でトンネルが再起動したときも同じ関数を呼ぶ。
import json, os, urllib.request

GIST_ID_DEFAULT = "2b44ac8cb115012a31d3c46661af933f"
GIST_OWNER = "yamahei21python"
GIST_ID = os.environ.get("GIST_ID", "").strip()
if not GIST_ID:
    try:
        from kaggle_secrets import UserSecretsClient
        GIST_ID = (UserSecretsClient().get_secret("GIST_ID") or "").strip()
    except Exception:
        GIST_ID = ""
if not GIST_ID:
    GIST_ID = GIST_ID_DEFAULT


def _gh_token():
    tok = os.environ.get("GITHUB_TOKEN", "")
    if tok:
        return tok
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("GITHUB_TOKEN") or ""
    except Exception:
        return ""


def publish_url(u):
    """現在の公開URL を Gist に書く（URL が毎回変わるため）。"""
    tok = _gh_token()
    if not tok:
        print("ℹ GITHUB_TOKEN 未設定 → 固定URL配布はスキップ（接続情報セルを使う）")
        return False
    # Ollama 版は url.txt を使う。同じ gist を共有しても衝突しないよう別名にする。
    files = {"vllm_url.txt": {"content": u + "\n"}}
    body = json.dumps({"files": files}).encode()
    req = urllib.request.Request(
        "https://api.github.com/gists/" + GIST_ID,
        data=body,
        method="PATCH",
        headers={
            "Authorization": "Bearer " + tok,
            "Accept": "application/vnd.github+json",
            "Content-Type": "application/json",
        },
    )
    try:
        with urllib.request.urlopen(req, timeout=20) as r:
            print("✅ 固定URL配布を更新: HTTP " + str(r.status))
            return True
    except Exception as e:
        print("⚠ Gist 更新失敗（続行）: " + repr(e))
        return False


publish_url(public_url)
print("📌 固定取得先: https://gist.githubusercontent.com/"
      + GIST_OWNER + "/" + GIST_ID + "/raw/vllm_url.txt")


---
### ✅ 公開URL経由の疎通確認（vLLM 版）

UI / models / streaming / OpenAI互換 / GPU常駐 の5点。


In [ ]:
# 公開URL経由の疎通確認（vLLM の OpenAI 互換 API 前提）
import json, time as _time, urllib.request, urllib.error

try:
    with open("/tmp/public_url.txt") as f:
        public_url = f.read().strip()
except FileNotFoundError:
    fail("公開URLがありません。トンネルセルを実行してください。")

# ---- トンネル確立の取り直し ----
for i in range(10):
    try:
        with urllib.request.urlopen(f"{public_url}/healthz", timeout=10) as r:
            if r.status == 200:
                if i:
                    print(f"✅ トンネル確立をここで確認（{(i+1)*3}s）")
                break
    except Exception:
        pass
    _time.sleep(3)
else:
    print("⚠ /healthz に届きません。トンネル未確立の可能性（[1] で判定）")

lines = [
    f"PUBLIC_URL={public_url}",
    f"MODEL={SERVED_NAME}",
    f"TIME={_time.strftime('%Y-%m-%d %H:%M:%S')}",
    "",
]
failures = []


def _judge(idx, label, ok, detail):
    mark = "✅" if ok else "❌"
    print(f"{mark} {label}: {detail}")
    lines.append(f"[{idx}] {label} -> {'PASS' if ok else 'FAIL'}: {detail}")
    if not ok:
        failures.append(f"[{idx}] {label}: {detail}")


def _with_retry(fn, tries=6, delay=5):
    """一過性の DNS/経路瞬断を吸収する（HTTP エラーは答えなので再試行しない）。"""
    last = None
    for i in range(max(1, tries)):
        try:
            return fn()
        except urllib.error.HTTPError:
            raise
        except Exception as e:
            last = e
            if i + 1 < tries:
                wait = delay * (i + 1)
                print(f"  ⚠ 一時的な通信エラー（{i+1}/{tries}）: {e} "
                      f"→ {wait}s 待って再試行")
                _time.sleep(wait)
    raise last


def _post(url, payload, timeout=600, stream=False):
    req = urllib.request.Request(
        url,
        data=json.dumps(payload).encode(),
        headers={"Content-Type": "application/json",
                 "ngrok-skip-browser-warning": "true"},
        method="POST",
    )
    return urllib.request.urlopen(req, timeout=timeout)


CHAT_BODY = {
    "model": SERVED_NAME,
    "messages": [{"role": "user", "content": "Say OK."}],
    "max_tokens": 64,
    "temperature": 0.7,
    "repetition_penalty": 1.05,
}

# [1] UI
print("[1] 認証なしで UI")
try:
    with _with_retry(lambda: urllib.request.urlopen(
            f"{public_url}/", timeout=60)) as r:
        html = r.read()
        ok_ui = (r.status == 200 and len(html) > 10000
                 and html.lstrip()[:15].lower().startswith(b"<!doctype html"))
        _judge(1, "ui", ok_ui,
               f"HTTP {r.status} {len(html)}B")
except Exception as e:
    _judge(1, "ui", False, f"{type(e).__name__}: {e}")

# [2] /v1/models（vLLM が応答している証明）
print("[2] /v1/models")
try:
    with _with_retry(lambda: urllib.request.urlopen(
            f"{public_url}/v1/models", timeout=60)) as r:
        md = json.loads(r.read())
    ids = [m.get("id") for m in md.get("data", [])]
    _judge(2, "models", r.status == 200 and SERVED_NAME in ids,
           f"HTTP {r.status} ids={ids}")
except Exception as e:
    _judge(2, "models", False, f"{type(e).__name__}: {e}")

# [3] ストリーミング（SSE がトンネルを抜けるか）
print("[3] ストリーミング")
try:
    body = dict(CHAT_BODY, stream=True)
    resp3 = _with_retry(lambda: _post(f"{public_url}/v1/chat/completions",
                                      body, timeout=600))
    times = []
    t0 = _time.time()
    with resp3 as r:
        for raw in r:            # 行到着 = 逐次受信の証拠
            if not raw.strip():
                continue
            times.append(_time.time() - t0)
    total = _time.time() - t0
    if len(times) >= 3:
        first = times[0]
        spread = times[-1] - first
        ok = spread > 0.2 and first < total * 0.8
        detail = (f"chunks={len(times)} first={first:.2f}s "
                  f"total={total:.2f}s spread={spread:.2f}s")
    else:
        ok, detail = False, f"chunks={len(times)} first=N/A"
    _judge(3, "streaming", ok, detail)
except Exception as e:
    _judge(3, "streaming", False, f"{type(e).__name__}: {e}")

# [4] OpenAI 互換（temperature=0 を送っても 200 で生成される）
#     プロキシの --enforce-sampling が 0.7 に矯正する経路の確認でもある。
print("[4] /v1/chat/completions (temperature=0)")
try:
    body4 = dict(CHAT_BODY, temperature=0, repetition_penalty=1.0)
    with _with_retry(lambda: _post(f"{public_url}/v1/chat/completions",
                                   body4, timeout=300)) as r:
        st4 = r.status
        cj = json.loads(r.read())
    ctext = "".join((c.get("message") or {}).get("content") or ""
                    for c in cj.get("choices", []))
    usage = cj.get("usage") or {}
    ok4 = st4 == 200 and bool(ctext or usage.get("completion_tokens"))
    _judge(4, "openai-compat", ok4,
           f"HTTP {st4} reply={ctext[:40]!r} usage={usage}")
except Exception as e:
    _judge(4, "openai-compat", False, f"{type(e).__name__}: {e}")

# [5] GPU 常駐（CPU フォールバック検出）
print("[5] GPU 常駐")
try:
    import subprocess as _sp
    mem = _sp.run(
        ["nvidia-smi", "--query-gpu=memory.used,memory.total",
         "--format=csv,noheader,nounits"],
        capture_output=True, text=True, timeout=30).stdout.strip()
    used = [int(x.split(",")[0]) for x in mem.splitlines() if "," in x]
    alive = _sp.run(["pgrep", "-f", "vllm serve"],
                    capture_output=True, text=True).stdout.strip() != ""
    _judge(5, "gpu-resident", alive and bool(used) and sum(used) > 4000,
           f"proc={'alive' if alive else 'DEAD'} vram={used}")
except Exception as e:
    _judge(5, "gpu-resident", False, f"{type(e).__name__}: {e}")

# ---- レポート ----
report = "\n".join(lines)
print()
print("=" * 60)
print(report)
print("=" * 60)
if not save_output("verify_result.txt", report + "\n"):
    failures.append("verify_result.txt を永続領域に保存できなかった")

if failures:
    fail("疎通確認に失敗: " + "; ".join(failures), TUNNEL_LOG)
print("✅ 5/5 パス — UI / models / streaming / OpenAI互換 / GPU常駐")


---
### 📋 接続情報


In [ ]:
# 接続情報（クライアント側の設定にコピペ）
import json

try:
    with open("/tmp/public_url.txt") as f:
        url = f.read().strip()
except FileNotFoundError:
    fail("公開URLがありません。トンネルセルを実行してください。")

try:
    with open("/tmp/vllm_cmd.json") as f:
        vinfo = json.load(f)
    max_len = vinfo.get("max_model_len")
except Exception:
    max_len = None

opencode_cfg = {
    "$schema": "https://opencode.ai/config.json",
    "provider": {
        "kaggle": {
            "npm": "@ai-sdk/openai-compatible",
            "name": "Kaggle vLLM",
            "options": {"baseURL": f"{url}/v1", "apiKey": "dummy"},
            "models": {SERVED_NAME: {"name": "Qwen3.8-27B-Uncensored (AWQ)"}},
        }
    },
    "model": f"kaggle/{SERVED_NAME}",
}

buf = [
    "=" * 64,
    f"  🌍 Public URL : {url}",
    f"  💬 チャット UI : {url}/",
    f"  🤖 Model      : {SERVED_NAME}",
    f"  📏 max_len    : {max_len}",
    "  🔓 認証        : なし（URL を知れば誰でも使える）",
    "  🎛 サンプリング : temperature<=0 → 0.7 / repetition_penalty → 1.05 に矯正",
    "=" * 64,
    "",
    "📋 curl:",
    f"  curl {url}/v1/models",
    "",
    "📋 Python（OpenAI 互換クライアント）:",
    "  from openai import OpenAI",
    "  client = OpenAI(",
    f"      base_url='{url}/v1',",
    "      api_key='dummy',",
    "  )",
    f"  client.chat.completions.create(model='{SERVED_NAME}', messages=[...])",
    "",
    "📋 opencode.json:",
    json.dumps(opencode_cfg, indent=2, ensure_ascii=False),
    "",
    "⚠ quick tunnel: 稼働保証なし。切れたらトンネルセルを再実行",
    "⚠ 公開URLは実行のたびに変わる。再実行後は接続情報を取り直す",
    "⚠ Kaggle: 9時間でカーネルが停止します",
]

report = "\n".join(buf)
print(report)
if not save_output("connection_info.txt", report + "\n"):
    fail("connection_info.txt を永続領域に保存できませんでした")


---
### 🔋 キープアライブ

既定 **5 分**（検証用）で自動終了します（`KEEPALIVE_MINUTES` で変更、上限 480）。

> 実行が完了しないと `/kaggle/working` の zip が生成されず、ログも成果物も
> 取得できません。検証は必ず完了させてから `kaggle kernels output` で行います。

```bash
kaggle kernels output yamahei21python/step2-vllm-qwen38-27b-ngrok -p out/
cat out/verify_result.txt out/connection_info.txt
```


In [ ]:
# キープアライブ: 有限ループ（完了させると /kaggle/working の zip を取得できる）
import json
import os
import subprocess
import time
import urllib.error
import urllib.request

KEEPALIVE_MINUTES = 2         # テスト用。完成したら 480 に戻す
MAX_MINUTES = 480
TUNNEL_CHECK_EVERY = 10

if KEEPALIVE_MINUTES <= 0:
    fail(f"KEEPALIVE_MINUTES={KEEPALIVE_MINUTES} は無効（1〜{MAX_MINUTES} で指定）")
if KEEPALIVE_MINUTES > MAX_MINUTES:
    print(f"⚠ KEEPALIVE_MINUTES={KEEPALIVE_MINUTES} → {MAX_MINUTES} に制限")
    KEEPALIVE_MINUTES = MAX_MINUTES

KEEPALIVE_CONF = "/tmp/keepalive.conf"
KEEPALIVE_KEEPALIVE = 60      # チャット送信ごとにリセットする時間（分）


def _read_keepalive_conf():
    """RUN=1/0 と MINUTES を読む（既定は 1 と KEEPALIVE_KEEPALIVE）。"""
    try:
        with open(KEEPALIVE_CONF) as f:
            lines = [l.strip() for l in f if l.strip() and "=" in l]
        d = dict(l.split("=", 1) for l in lines)
        run = int(d.get("RUN", 1))
        minutes = int(d.get("MINUTES", KEEPALIVE_KEEPALIVE))
    except FileNotFoundError:
        run, minutes = 1, KEEPALIVE_KEEPALIVE
    except Exception:
        run, minutes = 1, KEEPALIVE_KEEPALIVE
    if run <= 0:
        return 0, KEEPALIVE_KEEPALIVE
    if minutes <= 0:
        fail(f"KEEPALIVE_MINUTES={minutes} は無効（1〜{MAX_MINUTES} で指定）")
    if minutes > MAX_MINUTES:
        print(f"⚠ KEEPALIVE_MINUTES={minutes} → {MAX_MINUTES} に制限")
        minutes = MAX_MINUTES
    return run, minutes


try:
    with open("/tmp/public_url.txt") as f:
        url = f.read().strip()
except FileNotFoundError:
    url = ""


def _proc(name):
    """プロセス生死。vLLM は "vllm serve" の argv で照合する。"""
    try:
        if name == "vllm serve":
            args = ["pgrep", "-f", name]
        else:
            args = ["pgrep", "-x", name]
        out = subprocess.run(args, capture_output=True, text=True,
                             timeout=5).stdout.strip()
        return "alive" if out else "DEAD"
    except Exception:
        return "unknown"


def _gpu():
    try:
        return subprocess.run(
            ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=5,
        ).stdout.strip().replace("\n", " | ")
    except Exception:
        return "N/A"


print(f"Public URL: {url}")
RUN, KEEPALIVE_MINUTES = _read_keepalive_conf()
if RUN:
    print(f"🔋 ON。{KEEPALIVE_KEEPALIVE}分間維持（チャット送信でリセット、■ で停止）")
else:
    print("🛑 OFF（ RUN=0 ）。即時終了します")
    KEEPALIVE_MINUTES = 0
print("完了後: kaggle kernels output <slug> -p <dir> で成果物を取得")
print()

def _conf_mtime():
    """keepalive.conf の更新時刻。proxy がチャット送信ごとに書き直す。"""
    try:
        return os.path.getmtime(KEEPALIVE_CONF)
    except OSError:
        return 0.0


log = []
vllm_st = cf_st = "unknown"
cf_restarted = 0
tunnel_dead = 0
_restarted_now = False
# 1周も回らなかった場合（RUN=0 即時終了）は死活判定をしない。
# ループ未実行だと vllm_st/cf_st が "unknown" のままになり、
# 正常に動いているのに fail() する（実測バグ）。
ran_any = False
i = 0
# for ではなく while にする理由: proxy はチャット送信ごとに
# /tmp/keepalive.conf を MINUTES=60 で書き直す。for の range は開始時に
# 固定されるため、延長が効かない（実測バグ）。conf の mtime 変化で延ばす。
limit = KEEPALIVE_MINUTES
conf_mtime = _conf_mtime()
while i < limit:
    i += 1
    RUN, KEEPALIVE_MINUTES = _read_keepalive_conf()
    if not RUN:
        print(f"RUN=0 で {i}周目にキープアライブ終了")
        break
    _m = _conf_mtime()
    if _m != conf_mtime:
        conf_mtime = _m
        limit = i - 1 + KEEPALIVE_MINUTES
        print(f"♻ アクティビティ検出 → 残り {KEEPALIVE_MINUTES} 分に延長")
    ran_any = True
    time.sleep(60)
    vllm_st, cf_st = _proc("vllm serve"), _proc(TUNNEL_NAME)

    # トンネルが死んだら1回だけ起こす（URL が変わるので取り直す）
    if cf_st == "DEAD" and cf_restarted == 0:
        cf_restarted = 1
        try:
            subprocess.run(["pkill", "-x", TUNNEL_NAME], capture_output=True)
            time.sleep(1)
            with open(TUNNEL_LOG, "w") as _lg:
                subprocess.Popen(RESTART_CMD, stdout=_lg, stderr=_lg)
            _new_url = ""
            for _ in range(30):
                time.sleep(1)
                _new_url = probe_url()
                if _new_url:
                    break
            if _new_url:
                url = _new_url
                with open("/tmp/public_url.txt", "w") as _f:
                    _f.write(url + "\n")
                save_output("public_url.txt", url + "\n")
                print(f"♻ {TUNNEL_NAME} 再起動 → 新URL: {url}")
                if "publish_url" in globals():
                    publish_url(url)
                cf_st = "alive(restarted)"
                tunnel_dead = 0
                _restarted_now = True
        except Exception as _e:
            print(f"⚠ トンネル再起動失敗: {type(_e).__name__}: {_e}")

    line = (f"[{time.strftime('%H:%M:%S')}] "
            f"vllm={vllm_st} {TUNNEL_NAME}={cf_st} | GPU: {_gpu()}")

    if url and i % TUNNEL_CHECK_EVERY == 0 and not _restarted_now:
        try:
            with urllib.request.urlopen(f"{url}/healthz", timeout=15) as r:
                line += " | tunnel=OK" if r.status == 200 else f" | tunnel=HTTP{r.status}"
        except Exception as e:
            tunnel_dead += 1
            line += f" | tunnel=DEAD({type(e).__name__}) x{tunnel_dead}"
    _restarted_now = False

    log.append(line)
    print(line)

failed = False
if not ran_any:
    # 即時終了時は、判定材料が無いので現在の死活だけ 1 回確認する。
    # ここで DEAD でも「停止操作による終了」とは区別できないため、
    # 明示的な RUN=0 終了では fail させない（情報は keepalive.txt に残す）。
    vllm_st, cf_st = _proc("vllm serve"), _proc(TUNNEL_NAME)
    print(f"ℹ 即時終了のため死活判定はスキップ（vllm={vllm_st} "
          f"{TUNNEL_NAME}={cf_st}）")
else:
    failed = (vllm_st in ("DEAD", "unknown") or cf_st in ("DEAD", "unknown")
              or tunnel_dead >= 2)
report = "\n".join([
    f"result={'DEAD' if failed else ('stopped' if not RUN else 'alive')}",
    f"RUN={RUN}",
    f"minutes={KEEPALIVE_MINUTES}",
    f"ran_any={ran_any}",
    f"vllm={vllm_st}",
    f"tunnel={TUNNEL_NAME}:{cf_st}",
    f"url={url}",
    "",
] + log)
saved = save_output("keepalive.txt", report + "\n")
print()
print("⏱ キープアライブ終了。COMPLETE で /kaggle/working が zip 化されます")

# ============================================================
# 後片付け: GPU / プロセスを解放
# ============================================================
print()
print("=" * 56)
print("🧹 後片付け: GPU / プロセスを解放")
print("=" * 56)


def _pids(args):
    """pgrep の引数リストで PID を取る（pattern を必ず 1 要素で渡す）。

    shell 文字列で "pgrep -f vllm serve" と書くと、pgrep は positional を
    1 つしか取らないため "serve" が余計な引数になり失敗する（実測）。
    """
    try:
        r = subprocess.run(["pgrep"] + args, capture_output=True,
                           text=True, timeout=15)
        return [p for p in r.stdout.split() if p.strip()]
    except Exception:
        return []


# 順序: 外側（トンネル）→ 中継 → 推論本体（vLLM + EngineCore 子プロセス）
TARGETS = [
    (["-x", "ngrok"], "ngrok"),
    (["-x", "cloudflared"], "cloudflared"),
    (["-f", "auth_proxy.py"], "auth_proxy"),
    (["-f", "vllm serve"], "vllm"),
    (["-f", "VLLM::EngineCore"], "engine-core"),
]

for args, label in TARGETS:
    if _pids(args):
        subprocess.run(["pkill", "-TERM"] + args, capture_output=True)
        print(f"  TERM → {label}")
time.sleep(5)

for args, label in TARGETS:
    if _pids(args):
        subprocess.run(["pkill", "-KILL"] + args, capture_output=True)
        print(f"  KILL → {label}（残存していた）")

print()
time.sleep(2)
left = [label for args, label in TARGETS if _pids(args)]

if left:
    print(f"  ⚠ 残存: {', '.join(left)}")
else:
    print("  ✅ 全プロセス停止（vllm / engine-core / tunnel / proxy）")

save_output("cleanup.txt", "\n".join([
    f"leftover={','.join(left) if left else 'none'}",
    "",
]) + "\n")
print("  📄 cleanup.txt 保存済")
print("=" * 56)

if failed:
    fail("死活確認に失敗しました（keepalive.txt を確認）", "/tmp/vllm_1.log")
elif not saved:
    fail("keepalive.txt を保存できませんでした")
